In [1]:
import torch
import torch.distributions as TD
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import numpy as np
from datetime import datetime
import functools


# =========================
# Device
# =========================
enable_cuda = True
device = torch.device('cuda' if torch.cuda.is_available() and enable_cuda else 'cpu')


# =========================
# Parameters
# =========================
param = {
    "test": "power",                  # ['type1error', 'power']
    "sample_size": 200,               # 建议先 200 / 600
    "batch_size": 128,
    "z_dim": 1,
    "dx": 1,
    "dy": 1,
    "n_test": 50,
    "epochs_num": 100,
    "eps_std": 0.5,
    "dist_z": 'gaussian',
    "alpha_x": 0.25,                  # 在 binary weak alternative 里不直接使用，但保留接口
    "m_value": 100,
    "k_value": 2,                     # 论文建议 J=2 更有 power [file:3]
    "j_value": 500,                   # bootstrap 次数，先别太大
    "noise_dimension": 3,
    "hidden_layer_size": 64,
    "normal_ini": False,
    "preprocess": 'None',
    "G_lr": 1e-3,
    "alpha": 0.1,
    "alpha1": 0.05,
    "set_seeds": 42,
    "using_oracle": False,
    "lambda_1": 1.0,
    "lambda_2": 0.0,
    "using_Gen": '1',
    "boor_rv_type": 'gaussian',
    "wgt_decay": 1e-5,
    "lambda_3": 1e-5,
    "drop_out_p": 0.1,
    "M_train": 10,

    # 你的改进项，建议先关掉排错
    "lambda_mean": 0.0,
    "mean_samples": 20,

    "verbose_debug": False
}


# =========================
# Utilities
# =========================
def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)


def median_heuristic_l1(x):
    with torch.no_grad():
        n = x.shape[0]
        if n <= 1:
            return 1.0
        dist = torch.linalg.vector_norm(
            x.repeat(n, 1, 1) - torch.swapaxes(x.repeat(n, 1, 1), 0, 1),
            ord=1, dim=2
        )
        vals = dist[~torch.eye(n, dtype=torch.bool, device=dist.device)]
        med = torch.median(vals)
        med = med.item()
        if med <= 1e-8:
            med = 1.0
        return med


def bernoulli_from_probs(p, size):
    return np.random.choice(len(p), size=size, p=p)


# =========================
# Data generation
# =========================
def generate_samples_random(Ax, Ay, size=1000, sType='CI', dx=1, dy=1, dz=1,
                            nstd=0.05, alpha_x=0.05, preprocess="None",
                            dist_z='gaussian'):
    """
    Binary DGP for Section 4.3 weak alternative.

    H0 / CI:
        X, Y, Z are mutually independent Bernoulli(0.5).

    H1 / dependent:
        Z ~ Bernoulli(0.5)
        P(X,Y|Z=0) = [1/6, 1/3, 1/3, 1/6]
        P(X,Y|Z=1) = [1/3, 1/6, 1/6, 1/3]
    """
    num = size
    xy_arr = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])

    if sType in ['CI', 'type1error', 'H0']:
        z = np.random.binomial(1, 0.5, size=num)
        x = np.random.binomial(1, 0.5, size=num)
        y = np.random.binomial(1, 0.5, size=num)

    elif sType in ['dependent', 'power', 'H1']:
        z = np.random.binomial(1, 0.5, size=num)
        x = np.zeros(num)
        y = np.zeros(num)

        idx0 = np.where(z == 0)[0]
        idx1 = np.where(z == 1)[0]

        p_z0 = np.array([1/6., 1/3., 1/3., 1/6.])
        p_z1 = np.array([1/3., 1/6., 1/6., 1/3.])

        if len(idx0) > 0:
            xy0 = xy_arr[np.random.choice(len(xy_arr), size=len(idx0), p=p_z0)]
            x[idx0], y[idx0] = xy0[:, 0], xy0[:, 1]

        if len(idx1) > 0:
            xy1 = xy_arr[np.random.choice(len(xy_arr), size=len(idx1), p=p_z1)]
            x[idx1], y[idx1] = xy1[:, 0], xy1[:, 1]
    else:
        raise ValueError("sType must be one of ['CI','dependent','type1error','power','H0','H1']")

    X, Y, Z = x.reshape(-1, 1), y.reshape(-1, 1), z.reshape(-1, 1)

    if preprocess == "normalize":
        Z = (Z - Z.min()) / (Z.max() - Z.min() + 1e-8)
        X = (X - X.min()) / (X.max() - X.min() + 1e-8)
        Y = (Y - Y.min()) / (Y.max() - Y.min() + 1e-8)
    elif preprocess == "scale_Z":
        Z = Z / (Z.max() + 1e-8)
    elif preprocess == "None":
        pass

    X = torch.from_numpy(np.array(X)).float()
    Y = torch.from_numpy(np.array(Y)).float()
    Z = torch.from_numpy(np.array(Z)).float()
    return X, Y, Z



# =========================
# Statistic computation
# =========================
def get_p_value_stat_1(boot_num, M, n, gen_x_all_torch, gen_y_all_torch,
                       x_torch, y_torch, z_torch,
                       sigma_w, sigma_u=1, sigma_v=1,
                       boor_rv_type="gaussian"):
    w_mx = torch.linalg.vector_norm(
        z_torch.repeat(n, 1, 1) - torch.swapaxes(z_torch.repeat(n, 1, 1), 0, 1),
        ord=1, dim=2
    )
    w_mx = torch.exp(-w_mx / sigma_w)

    u_mx_1 = torch.exp(-torch.abs(y_torch.repeat(1, n) - y_torch.repeat(1, n).T) / sigma_u)
    u_mx_2 = torch.mean(
        torch.exp(
            -torch.abs(gen_y_all_torch.repeat(n, 1, 1) - y_torch.repeat(1, n).reshape(n, n, 1)) / sigma_u
        ),
        dim=2
    )
    u_mx_3 = u_mx_2.T

    gen_y_all_torch_rep = gen_y_all_torch.repeat(n, 1, 1)
    temp_mx = gen_y_all_torch_rep[:, :, 0].T
    sum_mx = torch.mean(
        torch.exp(-torch.abs(gen_y_all_torch_rep - temp_mx.reshape(n, n, 1)) / sigma_u),
        dim=2
    )
    for i in range(1, M):
        temp_mx = gen_y_all_torch_rep[:, :, i].T
        temp_add_mx = torch.mean(
            torch.exp(-torch.abs(gen_y_all_torch_rep - temp_mx.reshape(n, n, 1)) / sigma_u),
            dim=2
        )
        sum_mx = sum_mx + temp_add_mx

    u_mx_4 = 1.0 / M * sum_mx
    u_mx = u_mx_1 - u_mx_2 - u_mx_3 + u_mx_4

    v_mx_1 = torch.exp(-torch.abs(x_torch.repeat(1, n) - x_torch.repeat(1, n).T) / sigma_v)
    v_mx_2 = torch.mean(
        torch.exp(
            -torch.abs(gen_x_all_torch.repeat(n, 1, 1) - x_torch.repeat(1, n).reshape(n, n, 1)) / sigma_v
        ),
        dim=2
    )
    v_mx_3 = v_mx_2.T

    gen_x_all_torch_rep = gen_x_all_torch.repeat(n, 1, 1)
    temp2_mx = gen_x_all_torch_rep[:, :, 0].T
    sum2_mx = torch.mean(
        torch.exp(-torch.abs(gen_x_all_torch_rep - temp2_mx.reshape(n, n, 1)) / sigma_v),
        dim=2
    )
    for i in range(1, M):
        temp2_mx = gen_x_all_torch_rep[:, :, i].T
        temp2_add_mx = torch.mean(
            torch.exp(-torch.abs(gen_x_all_torch_rep - temp2_mx.reshape(n, n, 1)) / sigma_v),
            dim=2
        )
        sum2_mx = sum2_mx + temp2_add_mx

    v_mx_4 = 1.0 / M * sum2_mx
    v_mx = v_mx_1 - v_mx_2 - v_mx_3 + v_mx_4

    FF_mx = u_mx * v_mx * w_mx * (1 - torch.eye(n, device=device))
    stat = 1.0 / (n - 1) * torch.sum(FF_mx).item()

    if boor_rv_type == "rademacher":
        eboot = torch.sign(torch.randn(n, boot_num, device=device))
    elif boor_rv_type == "gaussian":
        eboot = torch.randn(n, boot_num, device=device)
    else:
        raise ValueError("boor_rv_type must be 'rademacher' or 'gaussian'")

    boottemp = np.zeros(boot_num)
    for bb in range(boot_num):
        random_mx = torch.matmul(
            eboot[:, bb].reshape(-1, 1),
            eboot[:, bb].reshape(-1, 1).T
        )
        bootmatrix = FF_mx * random_mx
        stat_boot = 1.0 / (n - 1) * torch.sum(bootmatrix).item()
        boottemp[bb] = stat_boot

    return stat, boottemp


# =========================
# Dataset
# =========================
class DatasetSelect(Dataset):
    def __init__(self, X, Y, Z):
        self.X_real = X
        self.Y_real = Y
        self.Z_real = Z
        self.sample_size = X.shape[0]

    def __len__(self):
        return self.sample_size

    def __getitem__(self, index):
        return self.X_real[index], self.Y_real[index], self.Z_real[index]


class DatasetSelect_GAN(torch.utils.data.Dataset):
    def __init__(self, X, Y, Z, batch_size):
        self.X_real = X
        self.Y_real = Y
        self.Z_real = Z
        self.batch_size = batch_size
        self.sample_size = X.shape[0]

    def __len__(self):
        return self.sample_size

    def __getitem__(self, index):
        return (
            self.X_real[index],
            self.Y_real[index],
            self.Z_real[index],
            self.Z_real[(self.batch_size + index) % self.sample_size]
        )


# =========================
# Noise sampler
# =========================
def sample_noise(sample_size, noise_dimension, noise_type, input_var):
    if noise_type == "normal":
        noise_generator = TD.MultivariateNormal(
            torch.zeros(noise_dimension).to(device),
            input_var * torch.eye(noise_dimension).to(device)
        )
        Z = noise_generator.sample((sample_size,))
    elif noise_type == "unif":
        Z = torch.rand(sample_size, noise_dimension, device=device)
    elif noise_type == "Cauchy":
        Z = TD.Cauchy(
            torch.tensor([0.0], device=device),
            torch.tensor([1.0], device=device)
        ).sample((sample_size, noise_dimension)).squeeze(2)
    else:
        raise ValueError("noise_type must be one of ['normal', 'unif', 'Cauchy']")
    return Z


# =========================
# Generator
# =========================
class Generator(torch.nn.Module):
    def __init__(self, input_dimension, output_dimension, noise_dimension,
                 hidden_layer_size, BN_type, ReLU_coef, drop_out_p,
                 drop_input=False):
        super(Generator, self).__init__()
        self.BN_type = BN_type
        self.drop_input = drop_input

        self.fc1 = torch.nn.Linear(input_dimension + noise_dimension, hidden_layer_size, bias=True)
        self.fc2 = torch.nn.Linear(hidden_layer_size, hidden_layer_size, bias=True)
        self.fc_last = torch.nn.Linear(hidden_layer_size, output_dimension, bias=True)

        if BN_type:
            self.BN1 = torch.nn.BatchNorm1d(hidden_layer_size, 0.8, affine=False)
            self.BN2 = torch.nn.BatchNorm1d(hidden_layer_size, 0.8, affine=False)

        self.leakyReLU1 = torch.nn.LeakyReLU(ReLU_coef)
        self.drop_out0 = torch.nn.Dropout(p=drop_out_p)
        self.drop_out1 = torch.nn.Dropout(p=drop_out_p)
        self.drop_out2 = torch.nn.Dropout(p=drop_out_p)
        self.sigmoid = torch.nn.Sigmoid()

    def forward(self, x):
        if self.drop_input:
            x = self.drop_out0(x)

        if self.BN_type:
            x = self.drop_out1(self.leakyReLU1(self.BN1(self.fc1(x))))
            x = self.drop_out2(self.leakyReLU1(self.BN2(self.fc2(x))))
        else:
            x = self.drop_out1(self.leakyReLU1(self.fc1(x)))
            x = self.drop_out2(self.leakyReLU1(self.fc2(x)))

        x = self.fc_last(x)
        x = self.sigmoid(x)
        return x


class NonFullyConnected_1(torch.nn.Module):
    def __init__(self, size_in, size_out, m, bias=True):
        super(NonFullyConnected_1, self).__init__()
        self.linear = torch.nn.Linear(m * size_in, m * size_out, bias=bias).to(device)
        self.mask = functools.reduce(
            torch.block_diag,
            [torch.ones(size_out, size_in) for _ in range(m)]
        ).to(device)

    def forward(self, x):
        self.linear.weight.data *= self.mask
        return self.linear(x)


class Generator_2(torch.nn.Module):
    def __init__(self, input_dimension, output_dimension, noise_dimension,
                 hidden_layer_size, BN_type, ReLU_coef,
                 hidden_layer_depth=1, ntargets_k=5):
        super(Generator_2, self).__init__()
        self.input_dimension = input_dimension + noise_dimension
        self.output_dimension = output_dimension
        self.ntargets_k = ntargets_k
        self.hidden_layer_sizes = [hidden_layer_size] * hidden_layer_depth
        self.BN_type = BN_type
        self.leakyrelu = torch.nn.LeakyReLU(ReLU_coef)

        self.linear_layers_from_input = torch.nn.Linear(
            self.input_dimension, ntargets_k * self.hidden_layer_sizes[0]
        )
        self.linear_layers_between = torch.nn.ModuleList([
            NonFullyConnected_1(self.hidden_layer_sizes[0], self.hidden_layer_sizes[0], ntargets_k)
            for _ in range(len(self.hidden_layer_sizes))
        ])
        self.linear8 = torch.nn.Linear(self.hidden_layer_sizes[0] * ntargets_k, self.output_dimension)
        self.sigmoid = torch.nn.Sigmoid()

        if BN_type:
            self.BN1 = torch.nn.BatchNorm1d(hidden_layer_size, 0.8, affine=False)

    def forward(self, input):
        if self.BN_type:
            output = self.linear_layers_from_input(input)
            output = self.leakyrelu(self.BN1(output))
            for linear_layers_between in self.linear_layers_between:
                output = linear_layers_between(output)
                output = self.leakyrelu(self.BN1(output))
        else:
            output = self.linear_layers_from_input(input)
            output = self.leakyrelu(output)
            for linear_layers_between in self.linear_layers_between:
                output = linear_layers_between(output)
                output = self.leakyrelu(output)

        output = self.linear8(output)
        output = self.sigmoid(output)
        return output


# =========================
# Loss
# =========================
def find_loss(y_torch, gen_y_all_torch, z_torch, sigma_w, sigma_u, M):
    n = z_torch.shape[0]

    w_mx = torch.linalg.vector_norm(
        z_torch.repeat(n, 1, 1) - torch.swapaxes(z_torch.repeat(n, 1, 1), 0, 1),
        ord=1, dim=2
    )
    w_mx = torch.exp(-w_mx / sigma_w)

    u_mx_1 = torch.exp(-torch.abs(y_torch.repeat(1, n) - y_torch.repeat(1, n).T) / sigma_u)
    u_mx_2 = torch.mean(
        torch.exp(
            -torch.abs(gen_y_all_torch.repeat(n, 1, 1) - y_torch.repeat(1, n).reshape(n, n, 1)) / sigma_u
        ),
        dim=2
    )
    u_mx_3 = u_mx_2.T

    gen_y_all_torch_rep = gen_y_all_torch.repeat(n, 1, 1)
    temp_mx = gen_y_all_torch_rep[:, :, 0].T
    sum_mx = torch.mean(
        torch.exp(-torch.abs(gen_y_all_torch_rep - temp_mx.reshape(n, n, 1)) / sigma_u),
        dim=2
    )

    for i in range(1, M):
        temp_mx = gen_y_all_torch_rep[:, :, i].T
        temp_add_mx = torch.mean(
            torch.exp(-torch.abs(gen_y_all_torch_rep - temp_mx.reshape(n, n, 1)) / sigma_u),
            dim=2
        )
        sum_mx = sum_mx + temp_add_mx

    u_mx_4 = 1.0 / M * sum_mx
    u_mx = u_mx_1 - u_mx_2 - u_mx_3 + u_mx_4

    FF_mx = u_mx * w_mx * (1 - torch.eye(n, device=device))
    loss = 1.0 / n * torch.sum(FF_mx)
    return loss


# =========================
# Training
# =========================
def train_ver3(X, Y, Z, X_test, Y_test, Z_test, M,
               noise_dimension, noise_type, G_lr, hidden_layer_size,
               DataLoader, BN_type, ReLU_coef,
               epochs_num=10, sigma_z=1, sigma_x=1, sigma_y=1,
               normal_ini=False,
               lambda_1=1, lambda_2=1, using_Gen='1', wgt_decay=0,
               lambda_3=0, drop_out_p=0.2, M_train=3,
               lambda_mean=0.0, mean_samples=20,
               verbose_debug=False):

    input_dimension = Z.shape[1]
    output_dimension_y = Y.shape[1]
    output_dimension_x = X.shape[1]

    if using_Gen == '1':
        G_zy = Generator(input_dimension, output_dimension_y, noise_dimension,
                         hidden_layer_size, BN_type, ReLU_coef, drop_out_p).to(device)
        G_zx = Generator(input_dimension, output_dimension_x, noise_dimension,
                         hidden_layer_size, BN_type, ReLU_coef, drop_out_p).to(device)
    elif using_Gen == '2':
        G_zy = Generator_2(input_dimension, output_dimension_y, noise_dimension,
                           hidden_layer_size, BN_type, ReLU_coef).to(device)
        G_zx = Generator_2(input_dimension, output_dimension_x, noise_dimension,
                           hidden_layer_size, BN_type, ReLU_coef).to(device)
    else:
        raise ValueError("using_Gen must be '1' or '2'")

    G_zy_solver = optim.Adam(G_zy.parameters(), lr=G_lr, betas=(0.5, 0.999), weight_decay=wgt_decay)
    G_zx_solver = optim.Adam(G_zx.parameters(), lr=G_lr, betas=(0.5, 0.999), weight_decay=wgt_decay)

    if normal_ini:
        for p in G_zy.parameters():
            p.data = torch.randn(p.shape, device=device, dtype=torch.float32) / np.sqrt(float(hidden_layer_size * 2))
        for p in G_zx.parameters():
            p.data = torch.randn(p.shape, device=device, dtype=torch.float32) / np.sqrt(float(hidden_layer_size * 2))

    for epoch in range(epochs_num):
        G_zy.train()
        G_zx.train()

        for X_real, Y_real, Z_real, Z_fake in DataLoader:
            X_real = X_real.to(device)
            Y_real = Y_real.to(device)
            Z_real = Z_real.to(device)

            batch_size = Z_real.shape[0]

            # ===== Y generator =====
            G_zy_solver.zero_grad()

            Z_real_repeat_y = Z_real.repeat(M_train, 1)
            Noise_fake_y = sample_noise(Z_real_repeat_y.shape[0], noise_dimension, noise_type, input_var=1.0 / 3.0).to(device)
            Y_fake_prob = G_zy(torch.cat((Z_real_repeat_y, Noise_fake_y), dim=1))
            Y_fake = Y_fake_prob.reshape(batch_size, M_train)

            mmd_loss_y = lambda_1 * find_loss(Y_real, Y_fake, Z_real, sigma_z, sigma_y, M_train)

            l1_y = 0.0
            for p in G_zy.parameters():
                l1_y += torch.linalg.vector_norm(p, ord=1)

            if lambda_mean > 0:
                Z_mean_repeat_y = Z_real.repeat_interleave(mean_samples, dim=0)
                Noise_mean_y = sample_noise(Z_mean_repeat_y.shape[0], noise_dimension, noise_type, input_var=1.0 / 3.0).to(device)
                Y_mean_fake_group = G_zy(torch.cat((Z_mean_repeat_y, Noise_mean_y), dim=1))
                Y_mean_fake = Y_mean_fake_group.reshape(batch_size, mean_samples, output_dimension_y).mean(dim=1)
                mean_loss_y = torch.nn.functional.mse_loss(Y_mean_fake, Y_real)
            else:
                mean_loss_y = torch.tensor(0.0, device=device)

            g_zy_error = mmd_loss_y + lambda_mean * mean_loss_y + lambda_3 * l1_y
            g_zy_error.backward()
            torch.nn.utils.clip_grad_norm_(G_zy.parameters(), max_norm=0.5)
            G_zy_solver.step()

            # ===== X generator =====
            G_zx_solver.zero_grad()

            Z_real_repeat_x = Z_real.repeat(M_train, 1)
            Noise_fake_x = sample_noise(Z_real_repeat_x.shape[0], noise_dimension, noise_type, input_var=1.0 / 3.0).to(device)
            X_fake_prob = G_zx(torch.cat((Z_real_repeat_x, Noise_fake_x), dim=1))
            X_fake = X_fake_prob.reshape(batch_size, M_train)

            mmd_loss_x = lambda_1 * find_loss(X_real, X_fake, Z_real, sigma_z, sigma_x, M_train)

            l1_x = 0.0
            for p in G_zx.parameters():
                l1_x += torch.linalg.vector_norm(p, ord=1)

            if lambda_mean > 0:
                Z_mean_repeat_x = Z_real.repeat_interleave(mean_samples, dim=0)
                Noise_mean_x = sample_noise(Z_mean_repeat_x.shape[0], noise_dimension, noise_type, input_var=1.0 / 3.0).to(device)
                X_mean_fake_group = G_zx(torch.cat((Z_mean_repeat_x, Noise_mean_x), dim=1))
                X_mean_fake = X_mean_fake_group.reshape(batch_size, mean_samples, output_dimension_x).mean(dim=1)
                mean_loss_x = torch.nn.functional.mse_loss(X_mean_fake, X_real)
            else:
                mean_loss_x = torch.tensor(0.0, device=device)

            g_zx_error = mmd_loss_x + lambda_mean * mean_loss_x + lambda_3 * l1_x
            g_zx_error.backward()
            torch.nn.utils.clip_grad_norm_(G_zx.parameters(), max_norm=0.5)
            G_zx_solver.step()

        if verbose_debug and ((epoch + 1) % 50 == 0 or epoch == 0 or epoch == epochs_num - 1):
            print(
                f"[Epoch {epoch+1}/{epochs_num}] "
                f"mmd_y={mmd_loss_y.item():.6f}, mean_y={mean_loss_y.item():.6f}, "
                f"mmd_x={mmd_loss_x.item():.6f}, mean_x={mean_loss_x.item():.6f}"
            )

    return G_zy, G_zx


# =========================
# Main experiment
# =========================
def mGAN(Ax, Ay, n=500, z_dim=1, simulation='type1error', batch_size=64, epochs_num=100,
         nstd=1.0, z_dist='gaussian', x_dims=1, y_dims=1, a_x=0.05, M=100, k=2, boot_num=500,
         noise_dimension=10, hidden_layer_size=64, normal_ini=False, preprocess='None',
         G_lr=1e-3, using_oracle=False, lambda_1=1, lambda_2=1, using_Gen='1',
         boor_rv_type="gaussian", wgt_decay=0, lambda_3=1e-5, drop_out_p=0.2, exp_num=0,
         M_train=10, lambda_mean=0.0, mean_samples=20, verbose_debug=False):

    if simulation == 'type1error':
        sim_x, sim_y, sim_z = generate_samples_random(
            Ax, Ay, size=n, sType='CI', dx=x_dims, dy=y_dims, dz=z_dim,
            nstd=nstd, alpha_x=a_x, dist_z=z_dist, preprocess=preprocess
        )
    elif simulation == 'power':
        sim_x, sim_y, sim_z = generate_samples_random(
            Ax, Ay, size=n, sType='dependent', dx=x_dims, dy=y_dims, dz=z_dim,
            nstd=nstd, alpha_x=a_x, dist_z=z_dist, preprocess=preprocess
        )
    else:
        raise ValueError('Test does not exist.')

    x, y, z = sim_x, sim_y, sim_z

    test_size = int(n / k)
    stat_all = np.zeros(k)
    boot_all = np.zeros((k, boot_num))

    for k_fold in range(k):
        k_fold_start = int(n / k * k_fold)
        k_fold_end = int(n / k * (k_fold + 1))

        X_test = x[k_fold_start:k_fold_end]
        Y_test = y[k_fold_start:k_fold_end]
        Z_test = z[k_fold_start:k_fold_end]

        X_train = torch.cat((x[:k_fold_start], x[k_fold_end:]), dim=0)
        Y_train = torch.cat((y[:k_fold_start], y[k_fold_end:]), dim=0)
        Z_train = torch.cat((z[:k_fold_start], z[k_fold_end:]), dim=0)

        if k == 1:
            X_train, Y_train, Z_train = X_test, Y_test, Z_test

        sigma_w_train = median_heuristic_l1(Z_train.to(device))
        sigma_u_train = median_heuristic_l1(Y_train.to(device))
        sigma_v_train = median_heuristic_l1(X_train.to(device))

        train_xyz = DatasetSelect_GAN(X_train, Y_train, Z_train, batch_size)
        DataLoader_xyz = torch.utils.data.DataLoader(train_xyz, batch_size=batch_size, shuffle=True)

        if not using_oracle:
            G_zy, G_zx = train_ver3(
                X=X_train, Y=Y_train, Z=Z_train, M=M,
                X_test=X_test, Y_test=Y_test, Z_test=Z_test,
                noise_dimension=noise_dimension, noise_type="normal",
                G_lr=G_lr, hidden_layer_size=hidden_layer_size,
                DataLoader=DataLoader_xyz, BN_type=False, ReLU_coef=0.1,
                epochs_num=epochs_num, sigma_z=sigma_w_train, sigma_x=sigma_v_train,
                sigma_y=sigma_u_train, normal_ini=normal_ini, lambda_1=lambda_1,
                lambda_2=lambda_2, using_Gen=using_Gen, wgt_decay=wgt_decay,
                lambda_3=lambda_3, drop_out_p=drop_out_p, M_train=M_train,
                lambda_mean=lambda_mean, mean_samples=mean_samples,
                verbose_debug=verbose_debug
            )

        dataset_test = DatasetSelect(X_test, Y_test, Z_test)
        dataloader_test = DataLoader(dataset_test, batch_size=1, shuffle=False)

        gen_x_all = torch.zeros(test_size, M)
        gen_y_all = torch.zeros(test_size, M)
        x_all = torch.zeros(test_size, x_dims)
        y_all = torch.zeros(test_size, y_dims)
        z_all = torch.zeros(test_size, z_dim)

        cur_itr = 0

        if using_oracle:
            for x_test, y_test, z_test in dataloader_test:
                z_scalar = int(z_test.item())
                if z_scalar == 0:
                    probs = np.array([1/6., 1/3., 1/3., 1/6.]) if simulation == 'power' else np.array([1/4., 1/4., 1/4., 1/4.])
                else:
                    probs = np.array([1/3., 1/6., 1/6., 1/3.]) if simulation == 'power' else np.array([1/4., 1/4., 1/4., 1/4.])

                xy_arr = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
                draws = xy_arr[bernoulli_from_probs(probs, M)]
                gen_x_all[cur_itr, :] = torch.tensor(draws[:, 0]).float()
                gen_y_all[cur_itr, :] = torch.tensor(draws[:, 1]).float()
                x_all[cur_itr, :] = x_test
                y_all[cur_itr, :] = y_test
                z_all[cur_itr, :] = z_test
                cur_itr += 1
        else:
            G_zx.eval()
            G_zy.eval()
            with torch.no_grad():
                for x_test, y_test, z_test in dataloader_test:
                    z_test_temp = z_test.repeat(M, 1).to(device)

                    noise_x = sample_noise(M, noise_dimension, "normal", input_var=1.0 / 3.0).to(device)
                    fake_x = G_zx(torch.cat((z_test_temp, noise_x), dim=1)).reshape(1, -1)

                    noise_y = sample_noise(M, noise_dimension, "normal", input_var=1.0 / 3.0).to(device)
                    fake_y = G_zy(torch.cat((z_test_temp, noise_y), dim=1)).reshape(1, -1)

                    gen_x_all[cur_itr, :] = fake_x.detach().cpu().reshape(-1)
                    gen_y_all[cur_itr, :] = fake_y.detach().cpu().reshape(-1)
                    x_all[cur_itr, :] = x_test
                    y_all[cur_itr, :] = y_test
                    z_all[cur_itr, :] = z_test
                    cur_itr += 1

        gen_x_all = gen_x_all.to(device)
        gen_y_all = gen_y_all.to(device)
        x_all = x_all.to(device)
        y_all = y_all.to(device)
        z_all = z_all.to(device)

        sigma_w = median_heuristic_l1(z_all)
        sigma_u = median_heuristic_l1(y_all)
        sigma_v = median_heuristic_l1(x_all)

        cur_stat, cur_boot = get_p_value_stat_1(
            boot_num, M, test_size,
            gen_x_all, gen_y_all, x_all, y_all, z_all,
            sigma_w=sigma_w, sigma_u=sigma_u, sigma_v=sigma_v,
            boor_rv_type=boor_rv_type
        )

        stat_all[k_fold] = cur_stat
        boot_all[k_fold, :] = cur_boot

        if verbose_debug:
            print(f"[Fold {k_fold+1}/{k}] stat={cur_stat:.6f}, "
                  f"sigma=({sigma_w:.4f},{sigma_u:.4f},{sigma_v:.4f}), "
                  f"gen_x_mean={gen_x_all.mean().item():.4f}, gen_y_mean={gen_y_all.mean().item():.4f}")

    final_stat = np.mean(stat_all)
    final_boot = np.mean(boot_all, axis=0)
    p_value = np.mean(final_boot >= final_stat)

    return final_stat, p_value


# =========================
# Repeated experiment
# =========================
def run_experiment(params):
    test = params["test"]
    sample_size = params["sample_size"]
    batch_size = params["batch_size"]
    z_dim = params["z_dim"]
    dx = params["dx"]
    dy = params["dy"]
    n_test = params["n_test"]
    epochs_num = params["epochs_num"]
    eps_std = params["eps_std"]
    dist_z = params["dist_z"]
    alpha_x = params["alpha_x"]
    m_value = params["m_value"]
    k_value = params["k_value"]
    j_value = params["j_value"]
    noise_dimension = params["noise_dimension"]
    hidden_layer_size = params["hidden_layer_size"]
    normal_ini = params["normal_ini"]
    preprocess = params["preprocess"]
    G_lr = params["G_lr"]
    alpha = params["alpha"]
    alpha1 = params["alpha1"]
    set_seeds = params["set_seeds"]
    using_oracle = params["using_oracle"]
    lambda_1 = params["lambda_1"]
    lambda_2 = params["lambda_2"]
    using_Gen = params["using_Gen"]
    boor_rv_type = params["boor_rv_type"]
    wgt_decay = params["wgt_decay"]
    lambda_3 = params["lambda_3"]
    drop_out_p = params["drop_out_p"]
    M_train = params["M_train"]
    lambda_mean = params["lambda_mean"]
    mean_samples = params["mean_samples"]
    verbose_debug = params["verbose_debug"]

    set_seed(set_seeds)

    a_f = torch.rand((z_dim, dx))
    Ax = a_f / torch.linalg.vector_norm(a_f, ord=1)

    a_g = torch.rand((z_dim, dy))
    Ay = a_g / torch.linalg.vector_norm(a_g, ord=1)

    p_values = np.array([])
    stats = np.array([])

    for n_iter in range(n_test):
        start_time = datetime.now()

        stat_value, p_value = mGAN(
            Ax=Ax, Ay=Ay, n=sample_size, z_dim=z_dim, simulation=test,
            batch_size=batch_size, epochs_num=epochs_num,
            nstd=eps_std, z_dist=dist_z, x_dims=dx, y_dims=dy,
            a_x=alpha_x, M=m_value, k=k_value, boot_num=j_value,
            noise_dimension=noise_dimension, hidden_layer_size=hidden_layer_size,
            normal_ini=normal_ini, preprocess=preprocess, G_lr=G_lr,
            using_oracle=using_oracle, lambda_1=lambda_1, lambda_2=lambda_2,
            using_Gen=using_Gen, boor_rv_type=boor_rv_type, wgt_decay=wgt_decay,
            lambda_3=lambda_3, drop_out_p=drop_out_p, exp_num=n_iter + 1,
            M_train=M_train, lambda_mean=lambda_mean, mean_samples=mean_samples,
            verbose_debug=verbose_debug
        )

        stats = np.append(stats, stat_value)
        p_values = np.append(p_values, p_value)

        elapsed = datetime.now() - start_time
        reject_alpha = np.mean(p_values < alpha)
        reject_alpha1 = np.mean(p_values < alpha1)

        print(f"--- Iteration {n_iter+1}/{n_test} took {elapsed} ---")
        print(f"stat = {stat_value:.6f}")
        print(f"p-value = {p_value:.6f}")

        if test == 'type1error':
            print(f"Type 1 error @ {alpha}: {reject_alpha:.4f}")
            print(f"Type 1 error @ {alpha1}: {reject_alpha1:.4f}")
        elif test == 'power':
            print(f"Power @ {alpha}: {reject_alpha:.4f}")
            print(f"Power @ {alpha1}: {reject_alpha1:.4f}")

    return stats, p_values


run_experiment(param)


--- Iteration 1/50 took 0:00:03.855234 ---
stat = 0.157880
p-value = 0.050000
Power @ 0.1: 1.0000
Power @ 0.05: 0.0000
--- Iteration 2/50 took 0:00:03.184843 ---
stat = 0.290449
p-value = 0.010000
Power @ 0.1: 1.0000
Power @ 0.05: 0.5000
--- Iteration 3/50 took 0:00:03.174162 ---
stat = 0.222479
p-value = 0.022000
Power @ 0.1: 1.0000
Power @ 0.05: 0.6667
--- Iteration 4/50 took 0:00:03.164645 ---
stat = 0.316180
p-value = 0.006000
Power @ 0.1: 1.0000
Power @ 0.05: 0.7500
--- Iteration 5/50 took 0:00:03.132698 ---
stat = 0.292990
p-value = 0.008000
Power @ 0.1: 1.0000
Power @ 0.05: 0.8000
--- Iteration 6/50 took 0:00:03.091054 ---
stat = 0.210418
p-value = 0.038000
Power @ 0.1: 1.0000
Power @ 0.05: 0.8333
--- Iteration 7/50 took 0:00:03.092817 ---
stat = 0.431273
p-value = 0.004000
Power @ 0.1: 1.0000
Power @ 0.05: 0.8571
--- Iteration 8/50 took 0:00:03.116528 ---
stat = 0.479189
p-value = 0.000000
Power @ 0.1: 1.0000
Power @ 0.05: 0.8750
--- Iteration 9/50 took 0:00:03.091263 ---
stat 

(array([0.15788016, 0.29044878, 0.22247859, 0.31618024, 0.29298979,
        0.21041795, 0.43127328, 0.47918942, 0.5235833 , 0.40678078,
        0.65760932, 0.29327464, 0.62385156, 0.90977663, 0.36917573,
        0.34090468, 0.59437561, 0.40138919, 0.51360038, 0.3987195 ,
        0.27724083, 0.78591589, 0.55795485, 0.37554427, 0.48195301,
        0.30578556, 0.10589222, 0.72250979, 0.30097549, 0.73896244,
        0.18657866, 0.77231182, 0.2397463 , 0.25282815, 0.49812675,
        0.60344702, 0.27746374, 1.19966342, 0.47716801, 0.58631635,
        0.16131972, 0.4591744 , 0.29671497, 0.2252807 , 0.29467548,
        0.27679065, 0.58777855, 0.5192869 , 0.39813171, 0.23276644]),
 array([0.05 , 0.01 , 0.022, 0.006, 0.008, 0.038, 0.004, 0.   , 0.002,
        0.   , 0.   , 0.004, 0.   , 0.   , 0.   , 0.02 , 0.   , 0.006,
        0.   , 0.   , 0.012, 0.   , 0.   , 0.01 , 0.002, 0.01 , 0.104,
        0.   , 0.01 , 0.   , 0.022, 0.   , 0.016, 0.026, 0.   , 0.   ,
        0.048, 0.   , 0.   , 0.   